# Cell-sphere simulator — backend diagnostics notebook
This notebook runs a few minimal simulations and produces diagnostic plots and sanity checks.

**Edit the import cell below** if your package/module names differ.


In [ ]:
# --- imports ---
import numpy as np
import math
import matplotlib.pyplot as plt

# If you have the simulator installed as a package, this should work:
try:
    from cell_sphere_sim.engine import SimulationEngine, SimParams
    from cell_sphere_sim.state import StateTable
    from cell_sphere_sim.fields.base import NullField
    from cell_sphere_sim.init import init_random_on_sphere
except Exception as e:
    print("Import failed. Edit this cell to match your repo structure.")
    raise

## Helpers: sphere plotting + basic diagnostics

In [ ]:
def unit(v, eps=1e-12):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.maximum(n, eps)

def project_to_sphere(x, R_E):
    return R_E * x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

def random_points_on_sphere(N, R_E, rng):
    g = rng.normal(size=(N,3))
    g = unit(g)
    return R_E * g

def random_tangent_polarity(x, R_E, rng):
    n = x / R_E
    g = rng.normal(size=x.shape)
    p = g - (np.sum(g*n, axis=1, keepdims=True) * n)
    return unit(p)

def sphere_mesh(R_E, n_lat=50, n_lon=100):
    # returns X,Y,Z on a grid for wireframe
    lats = np.linspace(0, np.pi, n_lat)
    lons = np.linspace(0, 2*np.pi, n_lon)
    latg, long = np.meshgrid(lats, lons, indexing="ij")
    X = R_E*np.sin(latg)*np.cos(long)
    Y = R_E*np.sin(latg)*np.sin(long)
    Z = R_E*np.cos(latg)
    return X,Y,Z

def plot_snapshot_3d(x, p, R_E, title="", stride=10, ax=None):
    from mpl_toolkits.mplot3d import Axes3D  # noqa
    if ax is None:
        fig = plt.figure(figsize=(7,7))
        ax = fig.add_subplot(111, projection="3d")
    X,Y,Z = sphere_mesh(R_E)
    ax.plot_wireframe(X, Y, Z, rstride=6, cstride=10, linewidth=0.3, alpha=0.25)
    ax.scatter(x[:,0], x[:,1], x[:,2], s=6, alpha=0.8)
    # polarity quivers (subsample)
    xs = x[::stride]
    ps = p[::stride]
    ax.quiver(xs[:,0], xs[:,1], xs[:,2], ps[:,0], ps[:,1], ps[:,2], length=R_E*0.08, normalize=True)
    ax.set_title(title)
    ax.set_box_aspect([1,1,1])
    return ax

def diagnostics_from_state(x, p, R_E):
    n = x / R_E
    radial_err = np.max(np.abs(np.linalg.norm(x, axis=1) - R_E))
    tangency = np.max(np.abs(np.sum(p*n, axis=1)))
    pnorm = np.max(np.abs(np.linalg.norm(p, axis=1) - 1.0))
    return {"max_radial_err": radial_err, "max_|p·n|": tangency, "max_| |p|-1 |": pnorm}

## Build a minimal configuration
Edit these numbers to match the scales you care about.

In [ ]:
rng = np.random.default_rng(0)

# --- states (K=2 example; set K=1..3 as desired) ---
state_table = StateTable(
    R=np.array([1.0, 1.0], dtype=float),
    Fm=np.array([1.0, 0.7], dtype=float),
    Dr=np.array([0.05, 0.03], dtype=float),
    fcil=np.array([2.0, 2.5], dtype=float),
    w=np.array([0.2, 0.5], dtype=float),
    lambda_div=np.array([0.0, 0.0], dtype=float),
    tau_div=np.array([0.0, 0.0], dtype=float),
)

params = SimParams(
    R_E=30.0,
    gamma_s=1.0,
    k_rep=5.0,
    alpha_dmin=0.2,
    eps=1e-6,
    dt=0.02,
    neighbor_radius_buffer=0.1,
    record_interval=10,
    division_enabled=False,
)

field = NullField(C=2)

## Scenario 1: no-contact persistence check
Sparse cells, check on-sphere constraint and polarity tangency. Also check that headings persist (up to diffusion) when there are no contacts.

In [ ]:
N = 300
state_id = rng.integers(0, 2, size=N, dtype=np.int32)
state_vars = np.zeros((N, 0), dtype=float)
x0, p0 = init_random_on_sphere(
    N=N,
    R_E=params.R_E,
    state_id=state_id,
    state_table=state_table,
    alpha_dmin=params.alpha_dmin,
    eps=params.eps,
    rng=rng,
)

engine = SimulationEngine(
    x=x0.copy(),
    p=p0.copy(),
    state_id=state_id.copy(),
    state_vars=state_vars.copy(),
    state_table=state_table,
    params=params,
    field=field,
    rng=np.random.default_rng(1),
)

T = 400
snapshots = []
diag = []
for step in range(T):
    t = step * float(params.dt)
    d = engine.step(t)
    if step % 50 == 0:
        snapshots.append((step, engine.x.copy(), engine.p.copy()))
    if step % 10 == 0:
        diag.append((step, diagnostics_from_state(engine.x, engine.p, params.R_E)))

diag[-1]

In [ ]:
# plot snapshots
fig = plt.figure(figsize=(14,6))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")
plot_snapshot_3d(snapshots[0][1], snapshots[0][2], params.R_E, title=f"step {snapshots[0][0]}", ax=ax1)
plot_snapshot_3d(snapshots[-1][1], snapshots[-1][2], params.R_E, title=f"step {snapshots[-1][0]}", ax=ax2)
plt.show()

# plot constraint diagnostics over time
steps = np.array([s for s,_ in diag])
rad = np.array([d["max_radial_err"] for _,d in diag])
pn = np.array([d["max_| |p|-1 |"] for _,d in diag])
tan = np.array([d["max_|p·n|"] for _,d in diag])

plt.figure()
plt.plot(steps, rad, label="max radial err")
plt.plot(steps, pn, label="max | |p|-1 |")
plt.plot(steps, tan, label="max |p·n|")
plt.yscale("log")
plt.legend()
plt.title("Constraint diagnostics (log scale)")
plt.show()

## Scenario 2: 2-cell CIL sanity check
Put two cells in contact and verify that the computed flee direction points away from the neighbor and that polarity relaxes toward it when `Dr=0`.

In [ ]:
# Override to Dr=0 to see deterministic relaxation
state_table2 = StateTable(
    R=state_table.R.copy(),
    Fm=state_table.Fm.copy(),
    Dr=np.array([0.0, 0.0], dtype=float),
    fcil=state_table.fcil.copy(),
    w=state_table.w.copy(),
    lambda_div=state_table.lambda_div.copy(),
    tau_div=state_table.tau_div.copy(),
)

params2 = params
params2.dt = 0.01

R_E = params2.R_E
Rcell = state_table2.R[0]
sigma = 2*Rcell

# Place two points on sphere close enough to be in contact
xA = np.array([R_E, 0, 0], dtype=float)
# move slightly along tangent direction toward +y then project
xB = project_to_sphere(np.array([R_E, 0.8*sigma, 0.0])[None,:], R_E)[0]

x0 = np.stack([xA, xB], axis=0)
p0 = random_tangent_polarity(x0, R_E, np.random.default_rng(3))
state_id = np.array([0,0], dtype=np.int32)
state_vars = np.zeros((2,0), dtype=float)

engine2 = SimulationEngine(
    x=x0.copy(), p=p0.copy(),
    state_id=state_id, state_vars=state_vars,
    state_table=state_table2, params=params2,
    field=NullField(C=2), rng=np.random.default_rng(4),
)

# Run a few steps; record dot(p, away-from-neighbor)
dots = []
for step in range(200):
    t = step * params2.dt
    engine2.step(t)
    x = engine2.x
    n = x / R_E
    # tangent direction toward neighbor for cell0
    dvec = x[1] - x[0]
    d_t = dvec - np.dot(dvec, n[0]) * n[0]
    d_t = d_t / np.linalg.norm(d_t)
    away = -d_t
    dots.append(np.dot(engine2.p[0], away))

plt.figure()
plt.plot(dots)
plt.ylim(-1.05, 1.05)
plt.title("Cell0 polarity · away-from-neighbor (Dr=0)")
plt.xlabel("step")
plt.ylabel("dot")
plt.show()

plot_snapshot_3d(engine2.x, engine2.p, R_E, title="2-cell contact (final)", stride=1)
plt.show()

## Scenario 3: moderate density + contact statistics
Run a denser simulation and track simple stats: mean speed proxy, contact counts, and minimum pair distance (approx via brute force on a subsample).

If your engine already outputs diagnostics dict keys (mean_speed, min_d, etc.), use those instead.

In [ ]:
N = 2500
state_id = rng.integers(0, 2, size=N, dtype=np.int32)
state_vars = np.zeros((N, 0), dtype=float)
x0, p0 = init_random_on_sphere(
    N=N,
    R_E=params.R_E,
    state_id=state_id,
    state_table=state_table,
    alpha_dmin=params.alpha_dmin,
    eps=params.eps,
    rng=rng,
)

engine3 = SimulationEngine(
    x=x0.copy(), p=p0.copy(),
    state_id=state_id.copy(), state_vars=state_vars.copy(),
    state_table=state_table, params=params,
    field=field, rng=np.random.default_rng(5),
)

mean_speed = []
mean_contacts = []
n_candidates = []
rad_err = []
tan_err = []
for step in range(300):
    t = step * float(params.dt)
    d = engine3.step(t)
    mean_speed.append(d.get("mean_speed", np.nan))
    mean_contacts.append(d.get("mean_contacts", np.nan))
    n_candidates.append(d.get("n_candidates", np.nan))
    dd = diagnostics_from_state(engine3.x, engine3.p, params.R_E)
    rad_err.append(dd["max_radial_err"])
    tan_err.append(dd["max_|p·n|"])

plt.figure()
plt.plot(mean_speed, label="mean speed")
plt.legend()
plt.title("Mean speed")
plt.show()

plt.figure()
plt.plot(mean_contacts, label="mean contacts/cell")
plt.legend()
plt.title("Contact stats")
plt.show()

plt.figure()
plt.plot(n_candidates, label="candidate pairs")
plt.legend()
plt.title("Neighbor candidate count")
plt.show()

plt.figure()
plt.plot(rad_err, label="max radial err")
plt.plot(tan_err, label="max |p·n|")
plt.yscale("log")
plt.legend()
plt.title("Constraint drift (log)")
plt.show()

plot_snapshot_3d(engine3.x, engine3.p, params.R_E, title="Dense snapshot (final)", stride=50)
plt.show()